# Accesorios: ¿crecer sin perder margen?
La responsable de Accesorios de Conecta quiere comparar Tienda y Web y revisar si el negocio mejoró frente al año anterior. Usaremos los mismos dos años de ventas del ejemplo: **noviembre de 2024 a octubre de 2026**.

Completarás la selección de registros y el margen bruto. Después usarás el dashboard para responder tres preguntas con cifras. Los controles y el diseño de los gráficos están preparados.

Los datos son simulados. Cada fila representa una venta única de un producto, con una o varias unidades. Los importes son COP sin IVA; no se incluyen devoluciones ni gastos operativos. La utilidad es bruta, no neta.

Ejecuta las celdas en orden con Shift + Enter. Necesitas pandas, numpy, Plotly e ipywidgets. Si falta alguna biblioteca, ejecuta `%pip install pandas numpy plotly ipywidgets` y reinicia la sesión. Los controles requieren un kernel activo.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import plotly
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from IPython.display import display
import ipywidgets as widgets

# Visor interactivo para JupyterLab.
pio.renderers.default = "plotly_mimetype"
px.defaults.template = "plotly_white"
pd.options.display.float_format = "{:,.2f}".format
print("Plotly", plotly.__version__)

## Una base lista para investigar
La generación de datos y los cálculos por venta están resueltos. `categoria` distingue Accesorios de Equipos; `canal` distingue Tienda de Web. `venta_neta` aplica los descuentos y `utilidad_bruta` resta el costo de los productos vendidos.

La semilla fija permite obtener los mismos datos cada vez que ejecutes el cuaderno.

In [ ]:
# Simulación reproducible: no son registros de una empresa real.
rng = np.random.default_rng(42)
catalogo = pd.DataFrame([
    ("Portátil", "Equipos", 2400000, 1980000),
    ("Teléfono", "Equipos", 1500000, 1125000),
    ("Tableta", "Equipos", 1100000, 770000),
    ("Monitor", "Equipos", 850000, 552500),
    ("Teclado", "Accesorios", 130000, 65000),
    ("Mouse", "Accesorios", 80000, 32000),
    ("Audífonos", "Accesorios", 180000, 81000),
    ("Webcam", "Accesorios", 200000, 100000),
], columns=["producto", "categoria", "precio_unitario", "costo_unitario"])
registros = []
for fecha in pd.date_range("2024-11-01", "2026-10-31"):
    mes = (fecha.year - 2024) * 12 + fecha.month - 11
    for canal in ["Tienda", "Web"]:
        # En Web crecen las operaciones y aumentan las rebajas durante el período.
        estacionalidad = 1.3 if fecha.month in [11, 12] else 1.0
        cantidad = int(rng.poisson((4 + mes * (0.22 if canal == "Web" else 0.07)) * estacionalidad))
        for _ in range(cantidad):
            producto = catalogo.iloc[int(rng.integers(len(catalogo)))]
            descuento = int(rng.choice([0, 5, 10]))
            if canal == "Web":
                descuento += min(mes // 4, 5)
            registros.append({
                "fecha": fecha, "canal": canal,
                **producto.to_dict(), "unidades": int(rng.integers(1, 4)),
                "descuento_pct": descuento
            })
ventas = pd.DataFrame(registros)
ventas.insert(0, "venta_id", [f"V{i:05d}" for i in range(1, len(ventas) + 1)])
display(ventas.head())
print("Registros:", len(ventas), "| Productos:", ventas["producto"].nunique())

ventas["venta_neta"] = ventas["unidades"] * ventas["precio_unitario"] * (1 - ventas["descuento_pct"] / 100)
ventas["cogs"] = ventas["unidades"] * ventas["costo_unitario"]
ventas["utilidad_bruta"] = ventas["venta_neta"] - ventas["cogs"]
ventas["mes"] = ventas["fecha"].dt.to_period("M").dt.to_timestamp()
MESES = pd.date_range(ventas["mes"].min(), ventas["mes"].max(), freq="MS")
CANALES = ["Todos", "Tienda", "Web"]
PERIODO = "Noviembre 2024–octubre 2026"

## Elegir la categoría, el canal y el período
Completa las tres líneas señaladas:

- Selecciona únicamente las filas cuya `categoria` sea `Accesorios`.
- Cuando el canal no sea `Todos`, conserva las filas de ese canal.
- Conserva las fechas anteriores a `limite`, que es el primer día del mes siguiente al mes final.

El inicio del rango ya está resuelto. El final debe incluir el mes completo: seleccionar septiembre significa incluir todas sus ventas, no solo las del día 1. Pistas: `.loc`, `.eq()` o `==`, y `<`.

In [ ]:
datos_categoria = ventas.iloc[0:0].copy()  # COMPLETAR: categoría Accesorios

def filtrar(datos, canal="Todos", inicio=None, fin=None):
    seleccion = datos.copy()
    if canal != "Todos":
        seleccion = seleccion.iloc[0:0]  # COMPLETAR: filtrar por canal
    if inicio is not None:
        seleccion = seleccion.loc[seleccion["fecha"] >= pd.Timestamp(inicio)]
    if fin is not None:
        limite = pd.Timestamp(fin) + pd.offsets.MonthBegin(1)
        seleccion = seleccion.iloc[0:0]  # COMPLETAR: fechas anteriores a limite
    return seleccion.copy()

print("Filas de Accesorios:", len(datos_categoria))

## Completar el margen de la selección
Calcula `margen` como **utilidad bruta total / ventas netas totales × 100**, siempre que los ingresos sean mayores que cero. En caso contrario conserva `None`.

Usa las variables `utilidad` e `ingresos`, que ya corresponden a los datos filtrados. La función prepara las series mensuales y los productos principales. Los meses sin ventas tienen suma cero y margen indefinido.

In [ ]:
def resumir(datos, meses=None):
    if meses is None:
        meses = MESES
    ingresos = datos["venta_neta"].sum()
    utilidad = datos["utilidad_bruta"].sum()
    margen = None  # COMPLETAR: porcentaje sobre los totales de la selección
    mensual = datos.groupby("mes")[["venta_neta", "utilidad_bruta"]].sum().reindex(meses, fill_value=0)
    # Un mes sin ventas tiene margen indefinido, no margen cero.
    mensual["margen_pct"] = mensual["utilidad_bruta"].div(mensual["venta_neta"].replace(0, np.nan)).mul(100)
    productos = datos.groupby("producto")["venta_neta"].sum().sort_values(ascending=False).head(5)
    return {"ingresos": ingresos, "utilidad": utilidad, "margen": margen,
            "registros": len(datos), "mensual": mensual, "productos": productos}

## Preparar la vista
La siguiente función recibe datos ya filtrados y construye cuatro tarjetas y tres gráficos. Utiliza directamente las series de pandas. No necesitas modificarla.

In [ ]:
def construir_dashboard(datos, etiqueta="Todo el catálogo", meses=None):
    if meses is None:
        meses = MESES
    periodo = f"{meses[0]:%Y-%m} a {meses[-1]:%Y-%m}"
    if datos.empty:
        fig = go.Figure()
        fig.add_annotation(text="Sin datos para esta selección", showarrow=False)
        fig.update_layout(template="plotly_white", title=etiqueta,
                          xaxis_visible=False, yaxis_visible=False)
        return fig

    fig = make_subplots(
        rows=3, cols=4,
        specs=[[{"type": "indicator"}] * 4,
               [{"type": "xy", "colspan": 2}, None, {"type": "xy", "colspan": 2}, None],
               [{"type": "xy", "colspan": 4}, None, None, None]],
        row_heights=[0.18, 0.38, 0.44], vertical_spacing=0.13,
        horizontal_spacing=0.10,
        subplot_titles=["", "", "", "", "Ventas por mes", "Margen bruto por mes", "Cinco productos con mayores ventas"]
    )
    r = resumir(datos, meses)
    # Millones en las tarjetas; COP completos disponibles en tablas y hover.
    indicadores = [
        ("Ventas netas · millones COP", r["ingresos"] / 1e6, ".1f", ""),
        ("Utilidad bruta · millones COP", r["utilidad"] / 1e6, ".1f", ""),
        ("Margen bruto", r["margen"], ".1f", "%"),
        ("Ventas registradas", r["registros"], ",.0f", "")
    ]
    for col, (titulo, valor, formato, sufijo) in enumerate(indicadores, 1):
        fig.add_trace(go.Indicator(
            mode="number", value=valor,
            title={"text": titulo, "font": {"size": 13}},
            number={"valueformat": formato, "suffix": sufijo, "font": {"size": 31}}), row=1, col=col)
    m = r["mensual"]
    fig.add_trace(go.Scatter(
        x=m.index, y=m["venta_neta"], mode="lines+markers",
        line={"color": "#007F5F", "width": 3}, name="Ventas netas",
        hovertemplate="%{x|%Y-%m}<br>Ventas: %{y:,.0f} COP<extra></extra>"), row=2, col=1)
    fig.add_trace(go.Scatter(
        x=m.index, y=m["margen_pct"], mode="lines+markers",
        line={"color": "#8052A3", "width": 3}, name="Margen bruto",
        hovertemplate="%{x|%Y-%m}<br>Margen: %{y:.2f}%<extra></extra>"), row=2, col=3)
    productos = r["productos"].sort_values()
    fig.add_trace(go.Bar(
        x=productos, y=productos.index, orientation="h",
        marker_color="#1677A5", name="Ventas por producto",
        hovertemplate="%{y}<br>Ventas: %{x:,.0f} COP<extra></extra>"), row=3, col=1)

    max_ventas = ventas.groupby("mes")["venta_neta"].sum().max()
    max_producto = ventas.groupby("producto")["venta_neta"].sum().max()
    fig.update_layout(
        title={"text": f"Conecta | {etiqueta}<br><sup>{periodo} · Datos simulados · Valores sin IVA</sup>", "x": 0.04, "font": {"size": 21}},
        template="plotly_white", height=850, showlegend=False,
        margin={"t": 120, "b": 55, "l": 115, "r": 45},
        font={"family": "Arial", "size": 13},

    )
    # Mismas escalas al cambiar canal: la comparación visual conserva su referencia.
    fig.update_yaxes(title_text="Ventas (COP)", range=[0, max_ventas * 1.15], row=2, col=1)
    fig.update_yaxes(title_text="Margen (%)", range=[0, 60], row=2, col=3)
    fig.update_xaxes(tickformat="%Y-%m", tickangle=-25, row=2, col=1)
    fig.update_xaxes(tickformat="%Y-%m", tickangle=-25, row=2, col=3)
    fig.update_xaxes(title_text="Ventas netas (COP)", range=[0, max_producto * 1.12], row=3, col=1)
    fig.update_yaxes(categoryorder="array", categoryarray=productos.index, row=3, col=1)
    return fig

## Explorar con los controles
Ejecuta esta celda después de completar los cálculos. Elige el canal y arrastra los extremos del rango mensual. Al soltar el control, tarjetas, series y productos se actualizan juntos.

Los controles empiezan en **abril–septiembre de 2026**. Si no aparecen datos, revisa las condiciones de la función de filtro. Usamos `responsive=False` para mantener estable la vista en Jupyter.

In [ ]:
config = {"displaylogo": False, "responsive": False}
control_canal = widgets.Dropdown(options=CANALES, value="Todos", description="Canal:")
control_fechas = widgets.SelectionRangeSlider(
    options=[(mes.strftime("%Y-%m"), mes) for mes in MESES],
    index=(17, 22), description="Meses:",
    continuous_update=False, layout=widgets.Layout(width="95%"),
    style={"description_width": "70px"}
)

def actualizar_vista(canal, rango):
    inicio, fin = rango
    seleccion = filtrar(datos_categoria, canal, inicio, fin)
    meses = pd.date_range(inicio, fin, freq="MS")
    print(f"Canal: {canal} | Desde {inicio:%Y-%m} hasta {fin:%Y-%m}, ambos incluidos")
    if seleccion.empty:
        print("No hay ventas en esta selección. Revisa la categoría y las condiciones del filtro.")
        return
    if resumir(seleccion, meses)["margen"] is None:
        print("Completa el margen bruto en resumir y vuelve a ejecutar esta celda.")
        return
    figura = construir_dashboard(seleccion, f"Accesorios | Canal: {canal}", meses)
    figura.show(renderer="plotly_mimetype", config=config)

salida_dashboard = widgets.interactive_output(
    actualizar_vista, {"canal": control_canal, "rango": control_fechas}
)
salida_dashboard.layout.min_height = "900px"
display(widgets.VBox([control_canal, control_fechas, salida_dashboard]))

## Contrastar los indicadores con una tabla
Esta tabla usa el rango seleccionado y presenta cada canal. Ejecútala después de mover las fechas. Las tarjetas redondean los importes a millones de COP; la tabla conserva COP completos.

La tabla calcula el margen directamente a partir de los totales, por lo que permite comprobar tu fórmula. Si el filtro de fechas está mal, debes corregirlo antes de interpretar los resultados.

In [ ]:
inicio, fin = control_fechas.value
ventas_periodo = filtrar(datos_categoria, "Todos", inicio, fin)
print(f"Período de la tabla: {inicio:%Y-%m} a {fin:%Y-%m}")
comparacion_canales = ventas_periodo.groupby("canal").agg(
    ventas_netas=("venta_neta", "sum"),
    utilidad_bruta=("utilidad_bruta", "sum"),
    ventas_registradas=("venta_id", "count")
)
comparacion_canales.loc["Todos"] = comparacion_canales.sum()
comparacion_canales["margen_pct"] = (
    100 * comparacion_canales["utilidad_bruta"]
    / comparacion_canales["ventas_netas"]
)
comparacion_canales["ventas_registradas"] = comparacion_canales["ventas_registradas"].astype(int)
display(comparacion_canales.rename(columns={
    "ventas_netas": "Ventas netas (COP)",
    "utilidad_bruta": "Utilidad bruta (COP)",
    "ventas_registradas": "Ventas registradas",
    "margen_pct": "Margen bruto (%)"
}).round(2))


**Respuesta 1.** Selecciona abril–septiembre de 2026. Compara Tienda y Web: anota ventas netas, margen bruto y cantidad de ventas registradas. ¿Qué canal vende más y cuál conserva mayor margen?

Tu respuesta: …

**Respuesta 2.** Selecciona Web y compara abril–septiembre de 2025 con abril–septiembre de 2026. Registra las ventas y el margen de cada período. Calcula el crecimiento de ventas con `(ventas_2026 / ventas_2025 − 1) × 100` y el cambio de margen con `margen_2026 − margen_2025`, en puntos porcentuales. ¿Por qué comparamos períodos de igual duración?

Tu respuesta: …

**Respuesta 3.** Recomienda una acción para Accesorios apoyada en dos cifras del análisis. Menciona un dato adicional que solicitarías antes de tomar la decisión. ¿Basta observar más ingresos para afirmar que el negocio mejoró?

Tu recomendación: …

Antes de entregar, verifica que los controles cambien todas las tarjetas y gráficos, que el título indique el canal y el período elegidos, y que tus respuestas incluyan cifras y unidades. Guarda el cuaderno con tus respuestas y vuelve a ejecutar los controles cuando lo abras.

[Controles interactivos de Jupyter](https://ipywidgets.readthedocs.io/en/stable/examples/Using%20Interact.html)